In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import sys

sys.path.insert(0, "/home/joshua/PhD_year_1/jaxsp/Adding_stellar_masses")

import jaxsp as jsp
from jaxsp.constants import GN

import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
import jax.numpy as jnp

import matplotlib.pyplot as plt

from scipy.interpolate import interp1d

import Stellar_sim_funcs as SSF

from collections import defaultdict

from scipy.special import sph_harm_y

# Regime-adaptive heating (arXiv:2510.17079)

Two heating mechanisms coexist:

**Quasi-particle (Eq 21)** — uncorrelated granules, valid in the outer halo:
$$\partial_t E = \frac{C\, G^2 \pi^3 \hbar^3 \rho_{\rm ULDM}^2}{m_a^3 \sigma^4}$$

**Soliton random walk (Eq 33)** — correlated fluctuations near the soliton:
$$\partial_t E = b\, \frac{G M_{\rm enc}^{\rm ULDM}\, \lambda_{\rm dB}}{R^2\, \tau_{\rm dB}}$$

At each $R$ we use whichever rate is larger, with Eq 33 forced inside the soliton core (Eq 21 diverges as $\sigma \to 0$ there and is unphysical).

**Radius evolution via $g$-inversion (Eq 38).** Build the energy-radius relation
- $U(R) = \int_0^R G M_{\rm enc}^{\rm DM}(r')/r'^2\, dr'$ (Eq 34)
- $T(R) = \tfrac{1}{2}\, G M_{\rm enc}^{\rm DM}(R)/R$ (Eq 35)
- $E_\ast(R) = -G M_\ast / R$ (stellar self-gravity; set $M_\ast=0$ in massless-star limit)
- $E(R) = U + T + E_\ast$ (Eq 36)

Monotonicity holds exactly, since
$$\frac{dE}{dR} = \frac{G M_{\rm enc}}{2 R^2} + 2\pi G \rho_{\rm ULDM} R + \frac{G M_\ast}{R^2} > 0,$$
so $g \equiv E^{-1}$ is well-defined. Evolution:
$$\frac{dR}{dt} = \frac{\partial_t E}{dE/dR}, \qquad R(t) = g\!\left(E_i + \int_0^t \partial_t E\, dt'\right).$$

Parameters: $b \approx 1/60$, $C \sim \mathcal{O}(1)$, $\sigma^2 = G M_{\rm enc}/R$, $\lambda_{\rm dB} = \hbar/(m_a \sigma)$, $\tau_{\rm dB} = \lambda_{\rm dB}/\sigma$, $\alpha = 1 + 4\pi R^3 \rho_{\rm ULDM}/M_{\rm enc}$ (analytic, from $dM_{\rm enc}/dR = 4\pi R^2 \rho$).


# at t = 0: 
# $R(t=0) = 0.19\text{Kpc}$

In [ ]:

m22 = 3
u = jsp.set_schroedinger_units(m22)
starting_point = 2 * u.from_Kpc


r_max_enclosing_frac = 0.99

In [ ]:


from scipy.integrate import cumulative_trapezoid

cNFWtides_params = jnp.array([
357964808.148399 * u.from_Msun,
25.690207,
0.407461,
0.012670 * u.from_Kpc,
1.857991 * u.from_Kpc,
3.729259
])

density_params = jsp.init_core_NFW_tides_params_from_sample(cNFWtides_params)

N = 512
rmin = .1 * u.from_pc
rmax = jsp.enclosing_radius(0.999, density_params)
potential_params = jsp.init_potential_params(density_params, rmin, rmax, N)

N = 1024
a = 1
b = 10

rmax = jsp.enclosing_radius(r_max_enclosing_frac, density_params)
eigenstate_lib = jsp.init_eigenstate_library(potential_params, rmin, rmax, a, b, N)

l = eigenstate_lib.radial_eigenmode_params.l
eigen_energies = eigenstate_lib.radial_eigenmode_params.E

rmin = 20 * u.from_pc

tol = 1e-7
wavefunction_params = jsp.init_wavefunction_params(eigenstate_lib, density_params, rmin, rmax, tol)

print('l max from jaxsp:', max(l))

r = np.logspace(np.log10(rmin), np.log10(rmax), 1000)

rho_psi = jax.vmap(jsp.rho_psi, in_axes=(0,None,None))
rho_ULDM_r = np.array(rho_psi(r, wavefunction_params, eigenstate_lib))

# M_enc via exact 1D integration (spherical harmonic orthogonality makes this equivalent
# to integrating the full 3D density, but O(Nr) instead of O(L^4))
M_enc = 4 * np.pi * cumulative_trapezoid(r**2 * rho_ULDM_r, r, initial=0)

plt.plot(r * u.to_Kpc, rho_ULDM_r * u.to_Msun / u.to_Kpc**3, label='ULDM density')
plt.plot(r * u.to_Kpc, M_enc * u.to_Msun, label='Enclosed mass')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('r (kpc)')
plt.ylabel(r'$\rho$ (Msun/kpc$^3$) or M(Msun)')
plt.title('ULDM density and enclosed mass')
plt.legend()
plt.show()

rho_ULDM_interp = interp1d(r, rho_ULDM_r, kind='cubic', fill_value="extrapolate")
M_enc_interp = interp1d(r, M_enc, kind='cubic', fill_value="extrapolate")

# Convert constants to code units
hbar_code = u.from_hbar
m_a_code = u.from_m22
G_code = GN.value * u.from_cm**3 / (u.from_g * u.from_s**2)


In [ ]:


print(M_enc_interp(starting_point) * u.to_Msun)
print(rho_ULDM_interp(starting_point) * u.to_Msun / u.to_Kpc**3)

print('dR/dt at starting point:', 1/120 * np.sqrt(G_code * M_enc_interp(starting_point)/starting_point) * u.to_Kpc / u.to_Gyr) 

In [ ]:
from scipy.integrate import solve_ivp, cumulative_trapezoid
from scipy.optimize import brentq

# =====================================================================
# Soliton core radius: rho(r_c) = rho(0)/2 via soliton-profile half-max
# =====================================================================
# rho(0) is the central (innermost-sampled) density. If the profile is
# sampled too coarsely near the origin the half-max estimate can drift;
# the fallback is the mass-radius scaling for ULDM solitons.
rho_0 = float(rho_ULDM_interp(r[0]))
try:
    r_c = brentq(lambda x: float(rho_ULDM_interp(x)) - rho_0 / 2.0,
                 float(r[0]), float(r[-1]))
except ValueError:
    M_halo_Msun = 357964808.0
    r_c = 1.6 * (M_halo_Msun / 1e9) ** (-1.0 / 3.0) * u.from_Kpc
print(f"Soliton core radius r_c = {r_c * u.to_Kpc:.3f} kpc")

# =====================================================================
# Energy-radius relation E(R) for g-inversion (Eqs 34-37 of 2510.17079)
# =====================================================================
M_stars = 0.0  # set to total stellar mass (code units) if self-gravity matters

R_E = np.logspace(np.log10(r[0]), np.log10(r[-1]), 4000)
M_E = np.asarray(M_enc_interp(R_E))
rho_E = np.asarray(rho_ULDM_interp(R_E))

U_arr = cumulative_trapezoid(G_code * M_E / R_E**2, R_E, initial=0.0)
T_arr = 0.5 * G_code * M_E / R_E
E_star_arr = -G_code * M_stars / R_E
E_arr = U_arr + T_arr + E_star_arr

# Analytic dE/dR using dM_enc/dR = 4 pi R^2 rho (avoids finite-difference noise)
dEdR_arr = (
    G_code * M_E / (2.0 * R_E**2)
    + 2.0 * np.pi * G_code * rho_E * R_E
    + G_code * M_stars / R_E**2
)

if not np.all(np.diff(E_arr) > 0):
    raise RuntimeError("E(R) is non-monotonic; check the density profile.")

E_of_R     = interp1d(R_E, E_arr,    kind='cubic', fill_value='extrapolate')
dEdR_of_R  = interp1d(R_E, dEdR_arr, kind='cubic', fill_value='extrapolate')
g_R_of_E   = interp1d(E_arr, R_E,    kind='cubic', fill_value='extrapolate',
                      bounds_error=False)

# =====================================================================
# Heating rates: keep lambda_dB and tau_dB explicit so the near-soliton
# rate remains correct for mixed ULDM+CDM halos (M_enc^ULDM != M_enc).
# =====================================================================
b_heat = 1.0 / 60.0
C_heat = 1.0

def _sigma(R_val, M):
    return np.sqrt(G_code * M / R_val)

def dE_dt_near_soliton(R_val):
    """Eq 33: soliton-random-walk heating."""
    M = float(M_enc_interp(R_val))
    M_ULDM = M  # pure ULDM here; replace with M_enc^ULDM(R) for mixed halos
    sigma = _sigma(R_val, M)
    lam_dB = hbar_code / (m_a_code * sigma)
    tau_dB = lam_dB / sigma
    return b_heat * G_code * M_ULDM * lam_dB / (R_val**2 * tau_dB)

def dE_dt_outer_halo(R_val):
    """Eq 21: uncorrelated-granule heating."""
    M = float(M_enc_interp(R_val))
    rho = float(rho_ULDM_interp(R_val))
    sigma = _sigma(R_val, M)
    return (C_heat * G_code**2 * np.pi**3 * hbar_code**3 * rho**2
            / (m_a_code**3 * sigma**4))

# =====================================================================
# ODE:  dR/dt = (dE/dt) / (dE/dR)    [Eq 38 in derivative form]
# Dominant mechanism wins at each R. Inside the soliton core the
# quasi-particle formula diverges (sigma -> 0), so force Eq 33 there.
# =====================================================================
r_min_interp = float(r[0])
r_max_interp = float(r[-1])

def dR_dt_paper(t, y):
    R_val = float(np.clip(y[0], r_min_interp, r_max_interp))
    M = float(M_enc_interp(R_val))
    if M <= 0:
        return [0.0]

    dE_ns = dE_dt_near_soliton(R_val)
    dE_oh = dE_dt_outer_halo(R_val) if R_val >= r_c else 0.0
    dEdt = max(dE_ns, dE_oh)

    dEdR = float(dEdR_of_R(R_val))
    return [dEdt / dEdR]

def hit_boundary(t, y):
    return r_max_interp - y[0]
hit_boundary.terminal = True
hit_boundary.direction = -1

total_evolve_time = 10 * u.from_Gyr
R0 = [float(starting_point)]

sol = solve_ivp(
    dR_dt_paper,
    (0, float(total_evolve_time)),
    R0,
    method='RK45',
    rtol=1e-8,
    atol=1e-12,
    max_step=float(0.01 * u.from_Gyr),
    events=hit_boundary,
    dense_output=True,
)

print(f"Integration: {sol.message}")
print(f"R(0)   = {R0[0] * u.to_Kpc:.4f} kpc")
print(f"R(end) = {sol.y[0, -1] * u.to_Kpc:.4f} kpc at t = {sol.t[-1] * u.to_Gyr:.2f} Gyr")

t_plot = np.linspace(0, sol.t[-1], 1000)
R_plot = sol.sol(t_plot)[0]


In [ ]:
# Crossover radius where the two heating rates are equal (informational)
try:
    R_cross = brentq(
        lambda R: dE_dt_outer_halo(R) - dE_dt_near_soliton(R),
        r_c, r_max_interp,
    )
    print(f"Eq 21 = Eq 33 at R_cross = {R_cross * u.to_Kpc:.3f} kpc")
except ValueError:
    R_cross = None

plt.plot(t_plot * u.to_Gyr, R_plot * u.to_Kpc)

plt.axhline(y=r_c * u.to_Kpc, color='r', linestyle='--', alpha=0.5,
            label=f'$r_c$ = {r_c * u.to_Kpc:.2f} kpc')
if R_cross is not None:
    plt.axhline(y=R_cross * u.to_Kpc, color='orange', linestyle='--', alpha=0.5,
                label=f'Eq 21 = Eq 33 at {R_cross * u.to_Kpc:.2f} kpc')

plt.xlabel('Time (Gyr)')
plt.ylabel('Orbital radius (kpc)')
plt.title('Estimated orbital radius evolution due to ULDM heating')
plt.legend()
plt.show()


the ρ²/m³ scaling in Eq 21 is only valid in a specific regime, and both "push m22 down" and "start central" take you out of that regime — so your simulation stops measuring the thing the formula describes.

regime needs $r \gg r_c$ which is essentially $\frac{r}{\lambda_{dB}} >> 1$ because the coredness is determined by the width of the ground state wavefunction (soliton).

Increasing m22 decreases the coredness of the ULDM and so r_c decreases making it easier to be in $r \gg r_c$

Also starting further out makes it such that $r \gg r_c$.



What's happening in D&K

Their UFDs sit at R_½/λ_dB ≳ 1 across the m22 values they test (~100–800)